# 06 — Scoring Evaluation & ChromaDB Population

## Objective

Evaluate the recommendation scoring system and populate ChromaDB for the recommendation engine.

Steps:
1. Validate recommendation_score distribution
2. Spot-check that scores make sense (flagships > budget)
3. Populate ChromaDB with engineered dataset
4. Test a sample retrieval query

**Input:** `data/processed/engineered_dataset.csv`

**Output:** `data/chroma_db/` (populated vector database)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.abspath("../src"))

pd.set_option("display.max_columns", None)

print("Libraries imported.")

In [ ]:
df = pd.read_csv("../data/processed/engineered_dataset.csv")
print(f"Loaded dataset shape: {df.shape}")

## 1. Recommendation Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df["recommendation_score"].plot(kind="hist", bins=30, ax=axes[0], color="#1a73e8", edgecolor="white")
axes[0].set_title("Recommendation Score Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Score")
axes[0].axvline(df["recommendation_score"].mean(), color="red", linestyle="--", label="Mean")
axes[0].legend()

# By segment
df.boxplot(column="recommendation_score", by="Target_Segment", ax=axes[1])
axes[1].set_title("Score by Segment", fontsize=12, fontweight="bold")
plt.suptitle("")

plt.tight_layout()
plt.show()

## 2. Score Validation — Sanity Checks

In [ ]:
# Average scores by segment
score_cols = ["recommendation_score", "performance_score", "camera_score",
              "battery_score", "display_score", "ai_score", "durability_score"]
segment_means = df.groupby("Target_Segment")[score_cols].mean().round(2)
print("Average scores by Target Segment:")
print(segment_means)

# Flagship should generally score higher
print("\n" + "="*50)
if "Flagship" in segment_means.index and "Budget" in segment_means.index:
    flagship_avg = segment_means.loc["Flagship", "recommendation_score"]
    budget_avg = segment_means.loc["Budget", "recommendation_score"]
    print(f"Flagship avg score: {flagship_avg:.2f}")
    print(f"Budget avg score: {budget_avg:.2f}")
    if flagship_avg > budget_avg:
        print("✅ Flagships score higher than Budget (expected)")
    else:
        print("⚠️ Budget scores higher than Flagship (unexpected)")

In [ ]:
# Top 5 phones per segment
for segment in df["Target_Segment"].unique():
    print(f"\nTop 5 {segment} phones:")
    top = df[df["Target_Segment"] == segment].nlargest(5, "recommendation_score")
    print(top[["Name", "Launch_Year", "Launch_Price", "recommendation_score"]].to_string(index=False))

## 3. Score by Series (2024+ only)

In [ ]:
latest = df[df["Launch_Year"] >= 2024]
if len(latest) > 0:
    print(f"2024+ phones: {len(latest)}")
    series_scores = latest.groupby("Series")["recommendation_score"].agg(["mean", "min", "max", "count"]).round(2)
    print(series_scores.sort_values("mean", ascending=False))
else:
    print("No 2024+ phones found.")

## 4. Populate ChromaDB

In [ ]:
from utils import get_chroma_client, get_or_create_collection, populate_chromadb

persist_dir = os.path.abspath("../data/chroma_db")
print(f"ChromaDB directory: {persist_dir}")

client = get_chroma_client(persist_dir)
collection = get_or_create_collection(client)

# Clear existing data to avoid duplicates
existing = collection.count()
if existing > 0:
    print(f"Clearing {existing} existing documents...")
    client.delete_collection("samsung_phones")
    collection = get_or_create_collection(client)

# Populate
populate_chromadb(df, collection)
print(f"\nChromaDB populated! Document count: {collection.count()}")

## 5. Test Sample Query

In [ ]:
# Test: query for flagship phones from 2024+
results = collection.query(
    query_texts=["best flagship Samsung phone with Galaxy AI"],
    n_results=5,
    where={"launch_year": {"$gte": 2024.0}}
)

print("Sample query: 'best flagship Samsung phone with Galaxy AI' (2024+)")
print("="*60)
for i, meta in enumerate(results["metadatas"][0]):
    print(f"{i+1}. {meta['name']} (Score: {meta['recommendation_score']:.2f}, Price: {meta['launch_price']:.0f} INR)")
print("="*60)